In [20]:
import numpy as np
from keras.models import Sequential
from keras.layers import GlobalAveragePooling2D,Conv2D,MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.preprocessing.image import ImageDataGenerator    
from keras.applications import MobileNetV2
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping
from sklearn.metrics import accuracy_score

In [21]:
train="C:\\Users\\adity\\Downloads\\archive (3)\\train"
test="C:\\Users\\adity\\Downloads\\archive (3)\\test"


In [22]:
emotions=['angry', 'happy', 'sad', 'surprise', 'neutral']
import tensorflow as tf


size=(128,128)


train_gen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input,
    validation_split=0.2,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)


train_data = train_gen.flow_from_directory(
    train,
    target_size=size,
    batch_size=32,
    class_mode='categorical',
    subset='training',
    shuffle=True,
    classes=emotions
)

# Validation data (20%)
val_data = train_gen.flow_from_directory(
    train,
    target_size=size,
    batch_size=32,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
    classes=emotions
)

# Test data (separate unseen dataset)
test_gen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input
)

test_data = test_gen.flow_from_directory(
    test,
    target_size=size,
    batch_size=32,
    class_mode='categorical',
    shuffle=False,
    classes=emotions
)

Found 19341 images belonging to 5 classes.
Found 4835 images belonging to 5 classes.
Found 6043 images belonging to 5 classes.


In [23]:
from tensorflow.keras.layers import Input
from tensorflow.keras.models import Model
inputs = Input(shape=(128, 128, 3))
base = MobileNetV2(weights='imagenet', include_top=False, input_tensor=inputs)

x = base.output
x = Conv2D(64, (3,3), activation='relu', padding='same')(x)
x = BatchNormalization()(x)
x = Conv2D(128, (3,3), activation='relu', padding='same')(x)
x = BatchNormalization()(x)
x = MaxPooling2D(pool_size=(2,2))(x)
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
x = Dense(128, activation='relu')(x)
outputs = Dense(5, activation='softmax')(x)

model = Model(inputs, outputs)

model.summary()

C:\Users\adity\AppData\Local\Temp\ipykernel_16588\3791187237.py:4: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base = MobileNetV2(weights='imagenet', include_top=False, input_tensor=inputs)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 64, 64,    │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 64, 64,    │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 64, 64,    │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 64, 64,    │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 64, 64,    │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 64, 64,    │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 64, 64,    │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 64, 64,    │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 64, 64,    │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 64, 64,    │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 64, 64,    │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 65, 65,    │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 32, 32,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 32, 32,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 32, 32,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 32, 32,    │      2,304 │ block_1_depthwis

 Total params: 3,087,109 (11.78 MB)

 Trainable params: 3,052,613 (11.64 MB)

 Non-trainable params: 34,496 (134.75 KB)

In [24]:
from keras.callbacks import ReduceLROnPlateau
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=1e-6)

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)
model.fit(
    train_data,
    epochs=20,
    validation_data=val_data,
    callbacks=[early_stopping, reduce_lr]
)

c:\Users\adity\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/20
605/605 ━━━━━━━━━━━━━━━━━━━━ 1093s 2s/step - accuracy: 0.4944 - loss: 1.3333 - val_accuracy: 0.4879 - val_loss: 1.6459 - learning_rate: 0.0010
Epoch 2/20
605/605 ━━━━━━━━━━━━━━━━━━━━ 725s 1s/step - accuracy: 0.6100 - loss: 1.1349 - val_accuracy: 0.3895 - val_loss: 1.9380 - learning_rate: 0.0010
Epoch 3/20
605/605 ━━━━━━━━━━━━━━━━━━━━ 593s 980ms/step - accuracy: 0.6370 - loss: 1.0862 - val_accuracy: 0.3195 - val_loss: 2.3405 - learning_rate: 0.0010
Epoch 4/20
605/605 ━━━━━━━━━━━━━━━━━━━━ 443s 732ms/step - accuracy: 0.6910 - loss: 0.9885 - val_accuracy: 0.6199 - val_loss: 1.1646 - learning_rate: 2.0000e-04
Epoch 5/20
605/605 ━━━━━━━━━━━━━━━━━━━━ 513s 848ms/step - accuracy: 0.7098 - loss: 0.9552 - val_accuracy: 0.6771 - val_loss: 1.0175 - learning_rate: 2.0000e-04
Epoch 6/20
605/605 ━━━━━━━━━━━━━━━━━━━━ 494s 816ms/step - accuracy: 0.7182 - loss: 0.9436 - val_accuracy: 0.6883 - val_loss: 0.9910 - learning_rate: 2.0000e-04
Epoch 7/20
605/605 ━━━━━━━━━━━━━━━━━━━━ 556s 918ms/step -

In [25]:
pred2=model.predict(test_data)

189/189 ━━━━━━━━━━━━━━━━━━━━ 34s 171ms/step


In [26]:
print(accuracy_score(test_data.labels, np.argmax(pred2, axis=1)))

0.7517789177560814


In [27]:
from sklearn.metrics import classification_report 
print(classification_report(test_data.classes,np.argmax(pred2, axis=1)))

              precision    recall  f1-score   support

           0       0.65      0.68      0.67       958
           1       0.89      0.88      0.88      1774
           2       0.66      0.61      0.63      1247
           3       0.88      0.83      0.85       831
           4       0.65      0.72      0.68      1233

    accuracy                           0.75      6043
   macro avg       0.75      0.74      0.74      6043
weighted avg       0.75      0.75      0.75      6043



In [28]:
model.save('mobilenet.keras')

In [1]:
import cv2
import time
import numpy as np
from collections import deque
import tensorflow as tf
from tensorflow.keras.models import load_model

# Avoid GPU memory pre-allocation (optional)
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except:
        pass

print("✅ Imports done")


✅ Imports done


In [30]:
# ===============================
# Cell 2: Load Model & Labels
# ===============================
# Path to your saved model (.h5)
model_path = "C:\\Users\\adity\\OneDrive\\Desktop\\ai moodmate\\mobilenet.keras"
# Load model
model = load_model(model_path)
print("✅ Model loaded:", model_path)

# Labels (FER2013 common order)
labels = ["angry","happy","sad","surprise","neutral"]

# Get model input details
input_shape = model.input_shape
img_size = input_shape[1]  # usually 48
channels = input_shape[-1] # 1 or 3
print("Model expects:", input_shape)


✅ Model loaded: C:\Users\adity\OneDrive\Desktop\ai moodmate\mobilenet.keras
Model expects: (None, 128, 128, 3)


In [33]:
def preprocess_face(face_img, img_size=48, model_channels=1):
    gray = cv2.cvtColor(face_img, cv2.COLOR_BGR2GRAY)
    resized = cv2.resize(gray, (img_size, img_size))
    arr = resized.astype("float32") / 255.0
    if model_channels == 1:
        arr = np.expand_dims(arr, -1)       # (48,48,1)
    else:
        arr = np.stack([arr, arr, arr], -1) # (48,48,3)
    return arr

def aggregate_probs(prob_deque, labels):
    if not prob_deque:
        return None, 0.0
    sum_probs = np.sum([p for (_, p) in prob_deque], axis=0)
    sum_probs = sum_probs / (np.sum(sum_probs) + 1e-9)
    idx = np.argmax(sum_probs)
    return labels[idx], float(sum_probs[idx])



In [34]:
import cv2
import pandas as pd
import numpy as np
import random
import time
from collections import deque

# Load your music dataset
music_df = pd.read_csv("C:\\Users\\adity\\Downloads\\muse_v3.csv")

# Define emotion-to-mood mapping
emotion_map = {
    "happy": ["happy", "energetic", "upbeat", "party"],
    "sad": ["sad", "melancholic", "emotional", "calm"],
    "angry": ["aggressive", "intense", "rock", "metal"],
    "surprise": ["exciting", "curious", "fast", "adventurous"],
    "neutral": ["chill", "relaxing", "balanced", "ambient"]
}
def recommend_music(emotion, n=20):
    """
    Recommend songs based on detected emotion by matching mood tags to genre or genre2.
    Prints track name, artist, genre(s), and lastfm link.
    """
    emotion = emotion.lower()
    tags = emotion_map.get(emotion, [])

    # Ensure genre columns exist and lowercase them for matching
    for col in ["genre", "genre2"]:
        if col in music_df.columns:
            music_df[col] = music_df[col].astype(str).str.lower()

    # Filter songs where genre or genre2 matches mood tags
    filtered = music_df[
        music_df["genre"].isin(tags) | music_df["genre2"].isin(tags)
    ] if "genre" in music_df.columns and "genre2" in music_df.columns else music_df

    recommendations = filtered.sample(min(len(filtered), n))

    print(f"\n🎧 Recommended tracks for emotion: {emotion.upper()}\n")
    for _, row in recommendations.iterrows():
        track = str(row.get("track", "Untitled")).strip()
        artist = str(row.get("artist", "Unknown Artist")).strip()
        genre1 = str(row.get("genre", "")).strip()
        genre2 = str(row.get("genre2", "")).strip()
        link = str(row.get("lastfm_url", "")).strip()

        genre_display = genre1 if genre1 and genre1 != "nan" else ""
        if genre2 and genre2 != "nan":
            genre_display += f" / {genre2}" if genre_display else genre2

        print(f"🎵 {track} by {artist} | Genre: {genre_display} | 🔗 {link}")
# Emotion detection setup
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
if face_cascade.empty():
    raise RuntimeError("❌ Haar cascade not loaded!")

cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("❌ Could not open webcam!")

interval = 2.0
prob_deque = deque()
num_frames = 200
frame_count = 0

print(f"✅ Webcam started — will stop after {num_frames} frames or press 'q' to quit")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.1, 5, minSize=(40,40))
    now = time.time()

    while prob_deque and (now - prob_deque[0][0] > interval):
        prob_deque.popleft()

    for (x,y,w,h) in faces:
        face_patch = frame[y:y+h, x:x+w]
        inp = preprocess_face(face_patch, img_size=img_size, model_channels=channels)
        inp_batch = np.expand_dims(inp, 0)

        probs = model.predict(inp_batch, verbose=0)[0]
        label = labels[np.argmax(probs)]
        conf  = np.max(probs)

        prob_deque.append((now, probs))

        cv2.rectangle(frame, (x,y), (x+w,y+h), (0,255,0), 2)
        cv2.putText(frame, f"{label} {conf*100:.1f}%", (x,y-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)

    agg_label, agg_conf = aggregate_probs(prob_deque, labels)
    if agg_label:
        cv2.putText(frame, f"Smoothed: {agg_label} {agg_conf*100:.1f}%",
                    (10,30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,0,0), 2)

    cv2.imshow("Emotion Detection", frame)

    if frame_count >= num_frames:
        print("✅ Reached frame limit, stopping...")
        break
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

# 🎧 Recommend music based on final emotion
if agg_label:
    print(f"\n🧠 Final detected emotion: {agg_label} ({agg_conf*100:.1f}%)")
    print(f"🎵 Recommended songs for emotion: {agg_label}\n")
    print(recommend_music(agg_label))
else:
    print("⚠️ No emotion detected — unable to recommend music.")

✅ Webcam started — will stop after 200 frames or press 'q' to quit
✅ Reached frame limit, stopping...

🧠 Final detected emotion: surprise (52.9%)
🎵 Recommended songs for emotion: surprise


🎧 Recommended tracks for emotion: SURPRISE

🎵 A Beautiful Day by Gym Class Heroes | Genre: hip-hop | 🔗 https://www.last.fm/music/gym%2bclass%2bheroes/_/a%2bbeautiful%2bday
🎵 Humko Tumse Pyaar Hai by Kumar Sanu | Genre: acoustic | 🔗 https://api.spotify.com/v1/tracks/7i1KXEFjxHSeti2WHBV9o5
🎵 Kamariya by Aastha Gill | Genre: rock | 🔗 https://api.spotify.com/v1/tracks/5cjVsWqIkBQC7acTRhL0RO
🎵 Roots (feat. Raja Kumari) by Divine | Genre: grunge | 🔗 https://api.spotify.com/v1/tracks/7KHz9ozYcZfGMn5IQxsy33
🎵 Dil Dhadakne Do by Farhan Akhtar | Genre: hip-hop | 🔗 https://api.spotify.com/v1/tracks/3MzNYNwL7I4dhG4sa1VkK1
🎵 Mazak Hai Kya by Emiway bantai | Genre: pop | 🔗 https://api.spotify.com/v1/tracks/42AnvmGwFt2ZwyROHffMCQ
🎵 Dhup Chik by Raftaar | Genre: dark cabaret | 🔗 https://api.spotify.com/v1/tracks/21

In [6]:
import torch
from transformers import pipeline
import pandas as pd

# Load music dataset
music_df = pd.read_csv("C:\\Users\\adity\\Downloads\\muse_v3.csv")

# Your emotion labels and genre mapping
emotion_labels = ["anger", "happy", "sad", "surprise", "neutral"]
emotion_map = {
    "angry": ["aggressive", "intense", "rock", "metal"],
    "happy": ["happy", "energetic", "upbeat", "party"],
    "sad": ["sad", "melancholic", "emotional", "calm"],
    "surprise": ["exciting", "curious", "fast", "adventurous"],
    "neutral": ["chill", "relaxing", "balanced", "ambient"]
}

# Load a multi-class emotion model
nlp_model = pipeline("text-classification", model="j-hartmann/emotion-english-distilroberta-base", framework="pt")

# Map model labels to your custom emotion labels
model_to_custom_map = {
    "anger": "anger",
    "joy": "happy",
    "sadness": "sad",
    "surprise": "surprise",
    "neutral": "neutral",
    "fear": "neutral",     # You can choose how to handle 'fear'
    "love": "happy"        # Optional: treat 'love' as 'happy'
}

def text_to_emotion(text):
    result = nlp_model(text)[0]
    label = result['label'].lower()
    score = result['score']
    print(f"\n🧠 Emotion detected: {label} ({score*100:.1f}%)")

    # Map to your custom label set
    mapped_label = model_to_custom_map.get(label, "neutral")
    if label != mapped_label:
        print(f"🔄 Mapped '{label}' to '{mapped_label}'")

    return mapped_label

def recommend_music(emotion, n=20):
    tags = emotion_map.get(emotion, [])

    genre_cols = [col for col in ["genre", "genre2"] if col in music_df.columns]
    for col in genre_cols:
        music_df[col] = music_df[col].astype(str).str.lower()

    filtered = music_df[
        pd.concat([music_df[col].isin(tags) for col in genre_cols], axis=1).any(axis=1)
    ] if genre_cols else music_df

    if filtered.empty:
        print("⚠️ No matching songs found. Showing random tracks instead.")
        filtered = music_df.sample(n=min(len(music_df), n))

    recommendations = filtered.sample(min(len(filtered), n))

    print(f"\n🎧 Recommended tracks for emotion: {emotion.upper()}\n")
    for _, row in recommendations.iterrows():
        track = str(row.get("track", "Untitled")).strip()
        artist = str(row.get("artist", "Unknown Artist")).strip()
        genre1 = str(row.get("genre", "")).strip()
        genre2 = str(row.get("genre2", "")).strip()
        link = str(row.get("lastfm_url", "")).strip()

        genre_display = genre1 if genre1 and genre1 != "nan" else ""
        if genre2 and genre2 != "nan":
            genre_display += f" / {genre2}" if genre_display else genre2

        print(f"🎵 {track} by {artist} | Genre: {genre_display} | 🔗 {link}")

# 📝 Run it
user_text = input("📝 Enter your mood or thoughts: ")
emotion = text_to_emotion(user_text)
recommend_music(emotion)

Device set to use cpu



🧠 Emotion detected: surprise (96.9%)
⚠️ No matching songs found. Showing random tracks instead.

🎧 Recommended tracks for emotion: SURPRISE

🎵 Drunk on Inconceivable Power by Circus of Bedlam | Genre: metal | 🔗 https://www.last.fm/music/circus%2bof%2bbedlam/_/drunk%2bon%2binconceivable%2bpower
🎵 Ladki Badi Anjani Hai by Alka Yagnik | Genre: ambient | 🔗 https://api.spotify.com/v1/tracks/6f3C6rJo7zvmfr1h5SRvxg
🎵 Gaata Rahe Mera Dil by Kishore Kumar | Genre: new wave | 🔗 https://api.spotify.com/v1/tracks/6knyRBZ0lLQoFV0611RiNX
🎵 Diwani Diwani - Chori chori Chupke Chupke / Soundtrack Version by Anu Malik | Genre: hip hop | 🔗 https://api.spotify.com/v1/tracks/3Y1fnsyoiiF1RDFPsJvDkc
🎵 Khoon Chala by Mohit Chauhan | Genre: industrial | 🔗 https://api.spotify.com/v1/tracks/56y6UhrAnXeSovZIE497DL
🎵 Aankh Marey (From "Simmba") by Kumar Sanu | Genre: electronica | 🔗 https://api.spotify.com/v1/tracks/63MvWd6T6yoS7h4AJ4Hjrm
🎵 Sultan by Sukhwinder Singh | Genre: dubstep | 🔗 https://api.spotify.com/v